# LeetCode 2148 — Count Elements With Strictly Smaller and Greater Elements bằng Python

**Link đề:** https://leetcode.com/problems/count-elements-with-strictly-smaller-and-greater-elements/
**Độ khó:** Easy · **Chủ đề:** Array, Sorting, Counting

```python
class Solution:
    def countElements(self, nums: list[int]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu chữ **"strictly" (nghiêm ngặt)**: `<` chứ không phải `<=`; và đếm theo **phần tử** (kể cả trùng giá trị).
2. Từ cách thử mọi cặp `O(n²)` → nhận ra chỉ cần **min** và **max** → `O(n)`.
3. Nắm `min`, `max`, `list.count`, `sum(1 for ...)`, so sánh chuỗi `a < x < b` của Python.
4. Xử lý edge case **mọi phần tử bằng nhau** khi dùng công thức `n - count(min) - count(max)`.
5. Viết thêm phiên bản **một lần duyệt** tự theo dõi min/max và số lần xuất hiện.

## 1. Đề bài

Cho mảng số nguyên `nums`. Đếm số **phần tử** `x` sao cho trong `nums` có **ít nhất một** phần tử **nhỏ hơn hẳn** `x` **và** ít nhất một phần tử **lớn hơn hẳn** `x`.

| `nums` | Kết quả | Giải thích |
|---|:-:|---|
| `[11,7,2,15]` | `2` | `7` (có 2 < 7 < 11) và `11` (có 7 < 11 < 15) |
| `[-3,3,3,90]` | `2` | hai phần tử `3` đều thỏa (có -3 và 90) — đếm **cả hai** |

**Constraints:** `1 <= len(nums) <= 100`, `-10^5 <= nums[i] <= 10^5`.

Câu hỏi then chốt: *với một `x`, cần biết gì về cả mảng để trả lời "có số nhỏ hơn và số lớn hơn không"?*

In [1]:
nums = [11, 7, 2, 15]
for x in nums:
    smaller = [y for y in nums if y < x]
    greater = [y for y in nums if y > x]
    print(f"x={x:>3}: nhỏ hơn {smaller}, lớn hơn {greater} -> {'đếm' if smaller and greater else 'bỏ'}")

x= 11: nhỏ hơn [7, 2], lớn hơn [15] -> đếm
x=  7: nhỏ hơn [2], lớn hơn [11, 15] -> đếm
x=  2: nhỏ hơn [], lớn hơn [11, 7, 15] -> bỏ
x= 15: nhỏ hơn [11, 7, 2], lớn hơn [] -> bỏ


## 2. Nền tảng Python cần biết

### 2.1. `min`, `max` trên list
`min(nums)` / `max(nums)` duyệt một lượt và trả giá trị nhỏ nhất / lớn nhất — mỗi hàm `O(n)`.

In [2]:
nums = [-3, 3, 3, 90]
print("min =", min(nums), "| max =", max(nums))

min = -3 | max = 90


### 2.2. So sánh "nghiêm ngặt" và so sánh chuỗi `a < x < b`

- `<` là **nhỏ hơn hẳn**; `<=` là nhỏ hơn **hoặc bằng**. Đề nói "strictly" → dùng `<`, `>`.
- Python cho phép viết `a < x < b`, nghĩa là `a < x and x < b`.

In [3]:
a, b = 2, 15
for x in (2, 7, 15):
    print(x, "| a < x < b:", a < x < b, "| a <= x <= b:", a <= x <= b)

2 | a < x < b: False | a <= x <= b: True
7 | a < x < b: True | a <= x <= b: True
15 | a < x < b: False | a <= x <= b: True


### 2.3. `list.count(v)` và `sum(1 for ...)`

- `nums.count(v)` đếm số lần giá trị `v` xuất hiện (`O(n)`).
- `sum(1 for x in nums if dieu_kien)` đếm số phần tử thỏa điều kiện. Trong Python `True` cũng cộng được như `1`, nên `sum(dieu_kien for x in nums)` cũng chạy.

In [4]:
nums = [-3, 3, 3, 90]
print(nums.count(3), nums.count(-3), nums.count(100))
print(sum(1 for x in nums if x > 0))
print(sum(x > 0 for x in nums), "<- True được cộng như 1")

2 1 0
3
3 <- True được cộng như 1


### 2.4. Biến cờ `any`

`any(iterable)` trả `True` nếu **có ít nhất một** phần tử truthy, dừng sớm khi gặp. Đọc giống tiếng Việt: "có số nào nhỏ hơn x không?" → `any(y < x for y in nums)`.

In [5]:
nums = [11, 7, 2, 15]
x = 2
print("có số < 2?", any(y < x for y in nums))
print("có số > 2?", any(y > x for y in nums))

có số < 2? False
có số > 2? True


### 2.5. Hiểu bài bằng tay — chữ "phần tử" vs "giá trị"

Đề đếm **phần tử**: `[-3, 3, 3, 90]` có hai phần tử giá trị `3`, mỗi phần tử đều thỏa → đếm `2`, không phải `1`.

Ai dùng `set(nums)` để bỏ trùng sẽ đếm sai. Kiểm chứng:

In [6]:
nums = [-3, 3, 3, 90]
lo, hi = min(nums), max(nums)
print("đếm theo phần tử:", sum(1 for x in nums if lo < x < hi))
print("đếm theo giá trị (SAI):", sum(1 for x in set(nums) if lo < x < hi))

đếm theo phần tử: 2
đếm theo giá trị (SAI): 1


## 3. Cách 1 — Brute force: với mỗi x, quét cả mảng tìm số nhỏ hơn và số lớn hơn

In [7]:
def count_elements_bruteforce(nums: list[int]) -> int:
    count = 0
    for x in nums:
        has_smaller = False
        has_greater = False
        for y in nums:
            if y < x:
                has_smaller = True
            if y > x:
                has_greater = True
        if has_smaller and has_greater:
            count += 1
    return count

print(count_elements_bruteforce([11, 7, 2, 15]))   # 2
print(count_elements_bruteforce([-3, 3, 3, 90]))   # 2
print(count_elements_bruteforce([5]))              # 0

2
2
0


### Chi phí và nút thắt

Hai vòng lồng nhau → `n · n = O(n²)` phép so sánh. Với `n = 100` chỉ 10.000 bước — chạy thừa nhanh. Nhưng ta **hỏi lại cùng một câu** cho mỗi `x`: "có ai nhỏ hơn x không?".

Đếm số phép so sánh:

In [8]:
def count_comparisons(nums):
    ops = 0
    for x in nums:
        for y in nums:
            ops += 2          # hai phép so sánh y < x, y > x
    return ops

for n in (4, 10, 100):
    print(f"n={n:>3}: {count_comparisons(list(range(n))):>6} phép so sánh")

n=  4:     32 phép so sánh
n= 10:    200 phép so sánh
n=100:  20000 phép so sánh


## 4. Insight: chỉ cần biết min và max

- "Có phần tử nhỏ hơn hẳn `x`" ⟺ **phần tử nhỏ nhất** `< x` ⟺ `min(nums) < x`.
  (Nếu có `y < x` thì `min ≤ y < x`; ngược lại nếu `min < x` thì chính `min` là phần tử nhỏ hơn.)
- "Có phần tử lớn hơn hẳn `x`" ⟺ `x < max(nums)`.

→ `x` được đếm ⟺ `min(nums) < x < max(nums)`.

Nói cách khác: **mọi phần tử đều được đếm, trừ những phần tử bằng min hoặc bằng max**.

In [9]:
nums = [11, 7, 2, 15]
lo, hi = min(nums), max(nums)
for x in nums:
    print(x, "->", lo < x < hi, "| brute force:", any(y < x for y in nums) and any(y > x for y in nums))

11 -> True | brute force: True
7 -> True | brute force: True
2 -> False | brute force: False
15 -> False | brute force: False


## 5. Cách 2 — Lời giải chính: tính min, max rồi đếm phần tử nằm giữa

Từng đoạn:

**Đoạn 1:** `lo = min(nums)`, `hi = max(nums)` — hai lượt duyệt `O(n)`.

**Đoạn 2:** duyệt `x`, nếu `lo < x < hi` thì `count += 1`.

**Đoạn 3:** trả `count`.

Chú ý: nếu mọi phần tử bằng nhau thì `lo == hi`, không có `x` nào thỏa `lo < x < hi` → tự động trả `0`. **Không cần xử lý riêng.**

In [10]:
def count_elements_minmax(nums: list[int]) -> int:
    lo, hi = min(nums), max(nums)      # Đoạn 1
    count = 0
    for x in nums:                      # Đoạn 2
        if lo < x < hi:
            count += 1
    return count                        # Đoạn 3

print(count_elements_minmax([11, 7, 2, 15]))
print(count_elements_minmax([-3, 3, 3, 90]))
print(count_elements_minmax([4, 4, 4]))

2
2
0


## 6. TRACE từng bước

`nums = [-3, 3, 3, 90, -3]` → `lo = -3`, `hi = 90`.

| `i` | `x` | `lo < x`? | `x < hi`? | Đếm? | `count` |
|:-:|:-:|:-:|:-:|:-:|:-:|
| 0 | -3 | ✗ (bằng min) | ✓ | không | 0 |
| 1 | 3 | ✓ | ✓ | **có** | 1 |
| 2 | 3 | ✓ | ✓ | **có** | 2 |
| 3 | 90 | ✓ | ✗ (bằng max) | không | 2 |
| 4 | -3 | ✗ | ✓ | không | 2 |

In [11]:
def trace_count_elements(nums):
    lo, hi = min(nums), max(nums)
    print(f"nums={nums}, lo={lo}, hi={hi}")
    count = 0
    for i, x in enumerate(nums):
        left, right = lo < x, x < hi
        if left and right:
            count += 1
        print(f"i={i}: x={x:>3} | lo<x: {left!s:<5} | x<hi: {right!s:<5} -> count={count}")
    return count

trace_count_elements([-3, 3, 3, 90, -3])

nums=[-3, 3, 3, 90, -3], lo=-3, hi=90
i=0: x= -3 | lo<x: False | x<hi: True  -> count=0
i=1: x=  3 | lo<x: True  | x<hi: True  -> count=1
i=2: x=  3 | lo<x: True  | x<hi: True  -> count=2
i=3: x= 90 | lo<x: True  | x<hi: False -> count=2
i=4: x= -3 | lo<x: False | x<hi: True  -> count=2


2

In [12]:
trace_count_elements([7, 7])     # mọi phần tử bằng nhau: lo == hi

nums=[7, 7], lo=7, hi=7
i=0: x=  7 | lo<x: False | x<hi: False -> count=0
i=1: x=  7 | lo<x: False | x<hi: False -> count=0


0

## 7. Code nộp LeetCode — Cách 2

In [13]:
class Solution:
    def countElements(self, nums: list[int]) -> int:
        lo, hi = min(nums), max(nums)
        return sum(1 for x in nums if lo < x < hi)

sol = Solution()
print(sol.countElements([11, 7, 2, 15]), sol.countElements([-3, 3, 3, 90]))

2 2


## 8. Cách 3 — Công thức `n - count(min) - count(max)` (và cái bẫy)

Hint của đề: mọi phần tử đều được đếm trừ các phần tử bằng min hoặc max, nên đáp án = `n - count(min) - count(max)`.

**Bẫy:** nếu mọi phần tử bằng nhau, `min == max`, ta trừ **hai lần** cùng một nhóm → ra số âm!

In [14]:
def formula_naive(nums):
    lo, hi = min(nums), max(nums)
    return len(nums) - nums.count(lo) - nums.count(hi)

print(formula_naive([-3, 3, 3, 90]))   # 2 ✓
print(formula_naive([4, 4, 4]))        # 3 - 3 - 3 = -3 ✗

2
-3


In [15]:
def count_elements_formula(nums: list[int]) -> int:
    lo, hi = min(nums), max(nums)
    if lo == hi:
        return 0
    return len(nums) - nums.count(lo) - nums.count(hi)

print(count_elements_formula([4, 4, 4]), count_elements_formula([-3, 3, 3, 90]))

0 2


Một cách viết khác tránh nhánh `if`: `max(0, n - cnt_min - cnt_max)`. Nhưng hãy cẩn thận: nó đúng ở đây vì khi `lo == hi` ta có `n - n - n = -n < 0`; khi `lo != hi`, hai nhóm rời nhau nên kết quả luôn `≥ 0`.

Code nộp:

In [16]:
class Solution:
    def countElements(self, nums: list[int]) -> int:
        lo, hi = min(nums), max(nums)
        if lo == hi:
            return 0
        return len(nums) - nums.count(lo) - nums.count(hi)

print(Solution().countElements([4, 4, 4]), Solution().countElements([11, 7, 2, 15]))

0 2


### Cách 4 — Sắp xếp (phụ, để mở đường cho biến thể)

Sau khi sắp tăng dần, phần tử bằng min nằm ở **đầu**, bằng max nằm ở **cuối**. Đáp án là số phần tử ở giữa khác hai đầu.

In [17]:
def count_elements_sort(nums: list[int]) -> int:
    a = sorted(nums)
    return sum(1 for x in a if a[0] < x < a[-1])

print(count_elements_sort([11, 7, 2, 15]), count_elements_sort([-3, 3, 3, 90]))

2 2


`a[0]` là phần tử đầu (nhỏ nhất), `a[-1]` là phần tử **cuối** (chỉ số âm đếm từ cuối). Cách này `O(n log n)` — chậm hơn cách 2, nhưng gợi ý cho các biến thể kiểu "đếm phần tử có ít nhất `k` số nhỏ hơn".

## 9. Cách 5 — Một lần duyệt: tự theo dõi min, max và số lần xuất hiện

Không gọi `min`, `max`, `count` riêng; duyệt một lượt và cập nhật:
- gặp `x < lo` → min mới, `cnt_lo = 1`; `x == lo` → `cnt_lo += 1`;
- tương tự cho max.
Cuối cùng áp dụng công thức (cẩn thận `lo == hi`).

In [18]:
def count_elements_one_pass(nums: list[int]) -> int:
    lo = hi = nums[0]
    cnt_lo = cnt_hi = 0
    for x in nums:
        if x < lo:
            lo, cnt_lo = x, 1
        elif x == lo:
            cnt_lo += 1
        if x > hi:
            hi, cnt_hi = x, 1
        elif x == hi:
            cnt_hi += 1
    if lo == hi:
        return 0
    return len(nums) - cnt_lo - cnt_hi

print(count_elements_one_pass([11, 7, 2, 15]), count_elements_one_pass([-3, 3, 3, 90]), count_elements_one_pass([4, 4]))

2 2 0


Chú ý `lo = hi = nums[0]` gán **cùng giá trị** cho hai biến (số nguyên bất biến nên an toàn). Ta khởi tạo đếm bằng `0` và để vòng lặp đếm lại chính `nums[0]` ở bước đầu (nhánh `x == lo`).

Trace nhỏ để chắc phần đếm:

In [19]:
def trace_one_pass(nums):
    lo = hi = nums[0]
    cnt_lo = cnt_hi = 0
    for x in nums:
        if x < lo:
            lo, cnt_lo = x, 1
        elif x == lo:
            cnt_lo += 1
        if x > hi:
            hi, cnt_hi = x, 1
        elif x == hi:
            cnt_hi += 1
        print(f"x={x:>3}: lo={lo:>3} (x{cnt_lo}), hi={hi:>3} (x{cnt_hi})")
    ans = 0 if lo == hi else len(nums) - cnt_lo - cnt_hi
    print("đáp án:", ans)
    return ans

trace_one_pass([3, -3, 90, 3, -3])

x=  3: lo=  3 (x1), hi=  3 (x1)
x= -3: lo= -3 (x1), hi=  3 (x1)
x= 90: lo= -3 (x1), hi= 90 (x1)
x=  3: lo= -3 (x1), hi= 90 (x1)
x= -3: lo= -3 (x2), hi= 90 (x1)
đáp án: 2


2

## 10. Vì sao đúng? (invariant)

**Bổ đề (Cách 2):** với mỗi phần tử `x`:
- tồn tại `y` trong `nums` với `y < x` ⟺ `min(nums) < x`;
- tồn tại `z` trong `nums` với `z > x` ⟺ `x < max(nums)`.

Chứng minh chiều ⇒: `min ≤ y < x`. Chiều ⇐: chọn `y = min`. Tương tự với max.

→ Đếm `x` thỏa `min < x < max` chính là định nghĩa của đề.

**Cách 3:** khi `min < max`, tập "bằng min" và tập "bằng max" **rời nhau**, và mọi phần tử còn lại nằm giữa → đáp án `n - cnt_min - cnt_max`. Khi `min == max`, không ai thỏa → `0`.

**Invariant Cách 5:** sau khi xử lý `t` phần tử đầu, `lo`/`hi` là min/max của tiền tố đó và `cnt_lo`/`cnt_hi` là số lần chúng xuất hiện trong tiền tố. Gặp min mới thì mọi lần đếm cũ không còn là min → đặt lại `1`.

## 11. Độ phức tạp

Gọi `n = len(nums)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(n²)` | `O(1)` | dễ nghĩ nhất |
| 2. min/max + đếm | `O(n)` (3 lượt duyệt) | `O(1)` | lời giải chính |
| 3. Công thức count | `O(n)` (4 lượt) | `O(1)` | nhớ case `min == max` |
| 4. Sắp xếp | `O(n log n)` | `O(n)` | hữu ích cho biến thể |
| 5. Một lần duyệt | `O(n)` (1 lượt) | `O(1)` | phù hợp dữ liệu dạng stream |

Không thể dưới `O(n)` vì phải nhìn mọi phần tử (một phần tử chưa xem có thể là min/max mới).

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

1. Hai ví dụ của đề.
2. `n = 1`, `n = 2` (không thể có phần tử ở giữa).
3. Mọi phần tử bằng nhau (bẫy công thức).
4. Nhiều phần tử trùng min/max.
5. Số âm, số 0.

In [20]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<24} PASS ({actual})")

impls = [count_elements_bruteforce, count_elements_minmax, count_elements_formula,
         count_elements_sort, count_elements_one_pass, Solution().countElements]
cases = [
    ("đề ví dụ 1", [11, 7, 2, 15], 2),
    ("đề ví dụ 2", [-3, 3, 3, 90], 2),
    ("n = 1", [5], 0),
    ("n = 2 khác nhau", [1, 2], 0),
    ("mọi phần tử bằng nhau", [4, 4, 4, 4], 0),
    ("trùng min và max", [1, 1, 2, 3, 3], 1),
    ("chỉ hai giá trị", [1, 2, 1, 2], 0),
    ("số âm và 0", [-5, 0, -1, 0, 5], 3),
    ("đã sắp xếp", [1, 2, 3, 4, 5], 3),
]
for name, nums, exp in cases:
    for f in impls:
        assert f(list(nums)) == exp, (name, f)
    check(name, count_elements_minmax(nums), exp)

đề ví dụ 1               PASS (2)
đề ví dụ 2               PASS (2)
n = 1                    PASS (0)
n = 2 khác nhau          PASS (0)
mọi phần tử bằng nhau    PASS (0)
trùng min và max         PASS (1)
chỉ hai giá trị          PASS (0)
số âm và 0               PASS (3)
đã sắp xếp               PASS (3)


In [21]:
import random
rng = random.Random(2148)
for _ in range(500):
    nums = [rng.randint(-3, 3) for _ in range(rng.randint(1, 12))]
    expected = count_elements_bruteforce(nums)
    for f in impls:
        assert f(list(nums)) == expected, (nums, f)
print("500 test ngẫu nhiên khớp brute force: PASS")

500 test ngẫu nhiên khớp brute force: PASS


## 13. Lỗi người mới hay gặp

### Lỗi 1: Dùng `<=` thay cho `<`

In [22]:
def wrong_non_strict(nums):
    lo, hi = min(nums), max(nums)
    return sum(1 for x in nums if lo <= x <= hi)

print("sai :", wrong_non_strict([11, 7, 2, 15]), "| đúng:", count_elements_minmax([11, 7, 2, 15]))

sai : 4 | đúng: 2


**Vì sao sai:** `lo <= x <= hi` luôn đúng với mọi phần tử → đếm tất cả. "Strictly" nghĩa là **nhỏ hơn hẳn**. **Sửa:** `lo < x < hi`.

### Lỗi 2: Bỏ trùng bằng `set` rồi đếm

In [23]:
def wrong_set(nums):
    lo, hi = min(nums), max(nums)
    return sum(1 for x in set(nums) if lo < x < hi)

print("sai :", wrong_set([-3, 3, 3, 90]), "| đúng:", count_elements_minmax([-3, 3, 3, 90]))

sai : 1 | đúng: 2


**Vì sao sai:** đề đếm **phần tử**, không đếm **giá trị khác nhau**. **Sửa:** duyệt trên `nums` gốc.

### Lỗi 3: Công thức trừ đôi khi `min == max`
Đã thấy ở Cách 3: `[4,4,4]` → `-3`. **Sửa:** `if lo == hi: return 0`.

### Lỗi 4: `n - 2` ("bỏ phần tử nhỏ nhất và lớn nhất")

In [24]:
def wrong_minus_two(nums):
    return max(0, len(nums) - 2)

for nums in ([11, 7, 2, 15], [1, 1, 2, 3, 3]):
    print(nums, "sai:", wrong_minus_two(nums), "| đúng:", count_elements_minmax(nums))

[11, 7, 2, 15] sai: 2 | đúng: 2
[1, 1, 2, 3, 3] sai: 3 | đúng: 1


**Vì sao sai:** min/max có thể xuất hiện **nhiều lần**. Ví dụ `[1,1,2,3,3]` chỉ có `2` ở giữa. **Sửa:** đếm theo điều kiện `lo < x < hi`.

### Lỗi 5: Khởi tạo min/max bằng `0`

In [25]:
def wrong_init(nums):
    lo, hi = 0, 0                    # sai: số đều âm hoặc đều dương sẽ hỏng
    for x in nums:
        lo = min(lo, x)
        hi = max(hi, x)
    return sum(1 for x in nums if lo < x < hi)

print("sai :", wrong_init([5, 6, 7]), "| đúng:", count_elements_minmax([5, 6, 7]))

sai : 2 | đúng: 1


**Vì sao sai:** `lo = 0` nhỏ hơn mọi số dương, nên `5` bị coi là "có số nhỏ hơn". **Sửa:** khởi tạo bằng `nums[0]` (hoặc `float('inf')` / `float('-inf')`).

## 14. Mẹo nhận dạng pattern

- "Tồn tại phần tử nhỏ hơn / lớn hơn" → thường chỉ cần **cực trị** (min/max), không cần so từng cặp.
- "Strictly" → `<`, `>`; đọc kỹ để biết đếm **phần tử** hay **giá trị khác nhau**.
- Công thức "tổng trừ phần bù" (`n - cnt_min - cnt_max`) → luôn kiểm tra trường hợp hai phần bù **trùng nhau**.
- Dữ liệu dạng luồng (stream) → giữ min/max và số đếm theo một lượt.

Bài liên quan: #1491 Average Salary Excluding the Minimum and Maximum Salary, #414 Third Maximum Number, #2089 Find Target Indices After Sorting Array.

## 15. Biến thể

### 15.1. Đếm phần tử có **ít nhất `k`** phần tử nhỏ hơn hẳn và ít nhất `k` lớn hơn hẳn
Sắp xếp rồi dùng `bisect` để đếm số phần tử `< x` và `> x`.

In [26]:
from bisect import bisect_left, bisect_right

def count_with_k_each_side(nums, k):
    a = sorted(nums)
    n = len(a)
    res = 0
    for x in nums:
        smaller = bisect_left(a, x)          # số phần tử < x
        greater = n - bisect_right(a, x)     # số phần tử > x
        if smaller >= k and greater >= k:
            res += 1
    return res

print(count_with_k_each_side([11, 7, 2, 15], 1))   # = bài gốc
print(count_with_k_each_side([1, 2, 3, 4, 5, 6, 7], 2))

2
3


`bisect_left(a, x)` trả vị trí đầu tiên có thể chèn `x` mà vẫn giữ `a` tăng dần = số phần tử `< x`. `bisect_right(a, x)` = số phần tử `<= x`.

In [27]:
a = [1, 3, 3, 3, 7]
print(bisect_left(a, 3), bisect_right(a, 3))

1 4


### 15.2. Trả về chính các phần tử được đếm (giữ thứ tự)

In [28]:
def middle_elements(nums):
    lo, hi = min(nums), max(nums)
    return [x for x in nums if lo < x < hi]

print(middle_elements([-3, 3, 3, 90, 5]))

[3, 3, 5]


## 16. Bài tập (TODO — tự làm, đáp án ở cuối)

### Bài 1 — Đoán output
1. `countElements([0, 0, 1])` = ?
2. `countElements([2, 1, 2, 3, 2])` = ?
3. `formula_naive([9, 9])` = ?

### Bài 2 — Trace tay Cách 5 với `[2, 5, 2, 8, 8]`: ghi `lo, cnt_lo, hi, cnt_hi` sau mỗi bước.

### Bài 3 — Điền chỗ trống (Cách 2)

In [29]:
def exercise_count_elements(nums: list[int]) -> int:
    # TODO 1: tính lo, hi
    # TODO 2: đếm x thỏa lo < x < hi
    pass

# assert exercise_count_elements([11, 7, 2, 15]) == 2
# assert exercise_count_elements([4, 4, 4]) == 0

### Bài 4 — Tự viết không dùng `min`, `max`, `sorted`
Duyệt một lần tìm `lo`, `hi`, rồi duyệt lần hai để đếm.

In [30]:
def exercise_count_no_builtin(nums: list[int]) -> int:
    # TODO
    pass

# assert exercise_count_no_builtin([-3, 3, 3, 90]) == 2
# assert exercise_count_no_builtin([5, 6, 7]) == 1

### Bài 5 — Biến thể: trung bình lương bỏ min và max (kiểu LeetCode 1491)
Cho `salary` gồm các số **đôi một khác nhau**, `len >= 3`. Trả trung bình của các lương trừ lương thấp nhất và cao nhất.

In [31]:
def exercise_average_excluding(salary: list[int]) -> float:
    # TODO
    pass

# assert exercise_average_excluding([4000, 3000, 1000, 2000]) == 2500.0

### Bài 6 — Mini challenge: đếm phần tử có phần tử nhỏ hơn hẳn **ở bên trái** và lớn hơn hẳn **ở bên phải**
Tức đếm chỉ số `j` sao cho tồn tại `i < j < k` với `nums[i] < nums[j] < nums[k]`. Yêu cầu `O(n)`.

In [32]:
def exercise_count_middle_of_triplet(nums: list[int]) -> int:
    # TODO: prefix min từ trái, suffix max từ phải
    pass

# assert exercise_count_middle_of_triplet([1, 2, 3]) == 1
# assert exercise_count_middle_of_triplet([3, 2, 1]) == 0
# assert exercise_count_middle_of_triplet([2, 1, 5, 0, 4, 6]) == 2

### Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Tìm min, max trước | phần tử bằng min/max không đếm | `[2,1,2,3,2]`: lo=1, hi=3 |
| 2 | Lần đầu `x == lo` và `x == hi` | gặp 5 > hi → hi=5, cnt=1 | gặp 8 lần 2 → cnt_hi=2 |
| 3 | `lo, hi = min(nums), max(nums)` | `sum(1 for ...)` | `if lo < x < hi` |
| 4 | Khởi tạo bằng `nums[0]` | `if x < lo: lo = x` | hai vòng `for` riêng |
| 5 | Tổng trừ min trừ max | chia cho `n - 2` | `(sum(s) - min(s) - max(s)) / (len(s) - 2)` |
| 6 | Cần min **bên trái** từng j, max **bên phải** | mảng `left_min[j]`, `right_max[j]` | đếm j: `left_min[j-1] < nums[j] < right_max[j+1]` |

## 17. Đáp án tham khảo (tự làm trước rồi hãy xem)

**Bài 1:** `0` (lo=0, hi=1, không có số ở giữa); `3` (ba số `2`); `-2` (công thức ngây thơ: `2 - 2 - 2`).

**Bài 2:** chạy cell dưới (`trace_one_pass`) — kết quả `lo=2 (x2)`, `hi=8 (x2)`, đáp án `5 - 2 - 2 = 1`.

In [33]:
print(Solution().countElements([0, 0, 1]), Solution().countElements([2, 1, 2, 3, 2]), formula_naive([9, 9]))
trace_one_pass([2, 5, 2, 8, 8])

0 3 -2
x=  2: lo=  2 (x1), hi=  2 (x1)
x=  5: lo=  2 (x1), hi=  5 (x1)
x=  2: lo=  2 (x2), hi=  5 (x1)
x=  8: lo=  2 (x2), hi=  8 (x1)
x=  8: lo=  2 (x2), hi=  8 (x2)
đáp án: 1


1

In [34]:
def answer_count_elements(nums):
    lo, hi = min(nums), max(nums)
    return sum(1 for x in nums if lo < x < hi)


def answer_count_no_builtin(nums):
    lo = hi = nums[0]
    for x in nums:
        if x < lo:
            lo = x
        if x > hi:
            hi = x
    count = 0
    for x in nums:
        if lo < x < hi:
            count += 1
    return count


def answer_average_excluding(salary):
    return (sum(salary) - min(salary) - max(salary)) / (len(salary) - 2)


def answer_count_middle_of_triplet(nums):
    n = len(nums)
    if n < 3:
        return 0
    right_max = [0] * n
    right_max[-1] = nums[-1]
    for k in range(n - 2, -1, -1):
        right_max[k] = max(right_max[k + 1], nums[k])
    count = 0
    left_min = nums[0]
    for j in range(1, n - 1):
        if left_min < nums[j] < right_max[j + 1]:
            count += 1
        left_min = min(left_min, nums[j])
    return count


assert answer_count_elements([11, 7, 2, 15]) == 2
assert answer_count_elements([4, 4, 4]) == 0
assert answer_count_no_builtin([-3, 3, 3, 90]) == 2
assert answer_count_no_builtin([5, 6, 7]) == 1
assert answer_average_excluding([4000, 3000, 1000, 2000]) == 2500.0
assert answer_count_middle_of_triplet([1, 2, 3]) == 1
assert answer_count_middle_of_triplet([3, 2, 1]) == 0
assert answer_count_middle_of_triplet([2, 1, 5, 0, 4, 6]) == 2
for name, nums, exp in cases:
    assert answer_count_no_builtin(nums) == exp, name
print("Đáp án tham khảo: PASS toàn bộ test!")

Đáp án tham khảo: PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def countElements(self, nums: list[int]) -> int:
        lo, hi = min(nums), max(nums)
        return sum(1 for x in nums if lo < x < hi)
```

**Khung tư duy:** "tồn tại phần tử nhỏ hơn/lớn hơn" → thay bằng so sánh với **min/max** → đếm theo điều kiện.

**Checklist trước khi nộp:**
- [ ] Dùng `<` / `>` nghiêm ngặt?
- [ ] Đếm theo **phần tử** (không bỏ trùng)?
- [ ] Nếu dùng công thức trừ, đã xử lý `min == max`?
- [ ] `n = 1`, `n = 2` ra `0`?
- [ ] Khởi tạo min/max bằng phần tử đầu, không bằng `0`?